In [ ]:
import tkinter as tk
from tkinter import messagebox
import sqlite3
import random

# ================= DATABASE =================

conn = sqlite3.connect("scores.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS leaderboard(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    player_name TEXT,
    attempts INTEGER
)
""")

conn.commit()

# ================= GAME VARIABLES =================

secret_number = 0
attempts = 0

# ================= FUNCTIONS =================

def start_game():
    global secret_number, attempts

    player = entry_name.get().strip()

    if player == "":
        messagebox.showerror("Error", "Please enter your name!")
        return

    difficulty = difficulty_var.get()

    if difficulty == "Easy":
        max_num = 50
    elif difficulty == "Medium":
        max_num = 100
    else:
        max_num = 200

    secret_number = random.randint(1, max_num)
    attempts = 0

    lbl_status.config(
        text=f"🎯 Guess a number between 1 and {max_num}",
        fg="#00ff99"
    )

    entry_guess.delete(0, tk.END)


def check_guess():
    global attempts

    if secret_number == 0:
        messagebox.showwarning("Warning", "Start the game first!")
        return

    try:
        guess = int(entry_guess.get())
    except ValueError:
        messagebox.showerror("Error", "Enter a valid number!")
        return

    attempts += 1

    if guess < secret_number:
        lbl_status.config(
            text="📉 Too Low! Try Again",
            fg="orange"
        )

    elif guess > secret_number:
        lbl_status.config(
            text="📈 Too High! Try Again",
            fg="orange"
        )

    else:
        lbl_status.config(
            text=f"🏆 Correct! Attempts: {attempts}",
            fg="lightgreen"
        )

        cursor.execute(
            """
            INSERT INTO leaderboard(player_name, attempts)
            VALUES (?,?)
            """,
            (entry_name.get(), attempts)
        )

        conn.commit()

        messagebox.showinfo(
            "Winner",
            f"🎉 Congratulations {entry_name.get()}!\n\nYou guessed correctly in {attempts} attempts."
        )


def show_leaderboard():

    top = tk.Toplevel(root)
    top.title("Leaderboard")
    top.geometry("400x350")
    top.configure(bg="#1e1e2f")

    tk.Label(
        top,
        text="🏆 TOP PLAYERS",
        font=("Segoe UI", 18, "bold"),
        bg="#1e1e2f",
        fg="#00d4ff"
    ).pack(pady=10)

    cursor.execute("""
    SELECT player_name, attempts
    FROM leaderboard
    ORDER BY attempts ASC
    LIMIT 10
    """)

    records = cursor.fetchall()

    if not records:
        tk.Label(
            top,
            text="No Scores Yet!",
            bg="#1e1e2f",
            fg="white",
            font=("Segoe UI", 12)
        ).pack()
        return

    for i, row in enumerate(records, start=1):
        tk.Label(
            top,
            text=f"{i}. {row[0]}  ➜  {row[1]} Attempts",
            bg="#1e1e2f",
            fg="white",
            font=("Segoe UI", 11)
        ).pack(anchor="w", padx=20)


def show_profile():

    player = entry_name.get().strip()

    if player == "":
        messagebox.showerror("Error", "Enter player name!")
        return

    cursor.execute("""
    SELECT COUNT(*), MIN(attempts)
    FROM leaderboard
    WHERE player_name=?
    """, (player,))

    result = cursor.fetchone()

    games_played = result[0]
    best_score = result[1]

    if games_played == 0:
        messagebox.showinfo(
            "Profile",
            "No records found!"
        )
    else:
        messagebox.showinfo(
            "Player Profile",
            f"""
👤 Player: {player}

🎮 Games Played: {games_played}

🏆 Best Score: {best_score} Attempts
"""
        )


def reset_game():
    global secret_number, attempts

    secret_number = 0
    attempts = 0

    entry_guess.delete(0, tk.END)

    lbl_status.config(
        text="🔄 Game Reset. Start Again!",
        fg="yellow"
    )


# ================= MAIN WINDOW =================

root = tk.Tk()

root.title("🎯 Number Guessing Game")
root.geometry("700x650")
root.resizable(False, False)

# Colors
BG = "#1e1e2f"
FG = "white"

root.configure(bg=BG)

# ================= TITLE =================

title = tk.Label(
    root,
    text="🎯 NUMBER GUESSING GAME",
    font=("Segoe UI", 22, "bold"),
    bg=BG,
    fg="#00d4ff"
)

title.pack(pady=15)

# ================= PLAYER NAME =================

tk.Label(
    root,
    text="Player Name",
    font=("Segoe UI", 12),
    bg=BG,
    fg=FG
).pack()

entry_name = tk.Entry(
    root,
    font=("Segoe UI", 12),
    width=25
)

entry_name.pack(pady=5)

# ================= DIFFICULTY =================

tk.Label(
    root,
    text="Select Difficulty",
    font=("Segoe UI", 12),
    bg=BG,
    fg=FG
).pack()

difficulty_var = tk.StringVar()
difficulty_var.set("Medium")

difficulty_menu = tk.OptionMenu(
    root,
    difficulty_var,
    "Easy",
    "Medium",
    "Hard"
)

difficulty_menu.pack(pady=5)

# ================= START BUTTON =================

tk.Button(
    root,
    text="🚀 Start Game",
    command=start_game,
    bg="#4CAF50",
    fg="white",
    font=("Segoe UI", 11, "bold"),
    width=20
).pack(pady=10)

# ================= GUESS =================

tk.Label(
    root,
    text="Enter Your Guess",
    font=("Segoe UI", 12),
    bg=BG,
    fg=FG
).pack()

entry_guess = tk.Entry(
    root,
    font=("Segoe UI", 12),
    width=25
)

entry_guess.pack(pady=5)

# ================= SUBMIT =================

tk.Button(
    root,
    text="✅ Submit Guess",
    command=check_guess,
    bg="#2196F3",
    fg="white",
    font=("Segoe UI", 11, "bold"),
    width=20
).pack(pady=10)

# ================= STATUS =================

lbl_status = tk.Label(
    root,
    text="Welcome! Click Start Game",
    font=("Segoe UI", 12, "bold"),
    bg=BG,
    fg="#00ff99"
)

lbl_status.pack(pady=15)

# ================= LEADERBOARD =================

tk.Button(
    root,
    text="🏆 Leaderboard",
    command=show_leaderboard,
    bg="#9C27B0",
    fg="white",
    font=("Segoe UI", 11, "bold"),
    width=20
).pack(pady=5)

# ================= PROFILE =================

tk.Button(
    root,
    text="👤 Player Profile",
    command=show_profile,
    bg="#FF9800",
    fg="white",
    font=("Segoe UI", 11, "bold"),
    width=20
).pack(pady=5)

# ================= RESET =================

tk.Button(
    root,
    text="🔄 Reset Game",
    command=reset_game,
    bg="#FFC107",
    fg="black",
    font=("Segoe UI", 11, "bold"),
    width=20
).pack(pady=5)

# ================= EXIT =================

tk.Button(
    root,
    text="❌ Exit",
    command=root.quit,
    bg="#F44336",
    fg="white",
    font=("Segoe UI", 11, "bold"),
    width=20
).pack(pady=5)

# ================= RUN =================

root.mainloop()

conn.close()